# Filing freshness check

Pull one company's real SEC filing history and check whether its data is actually complete and current.

## Executive summary

*Write this section last, after finishing the analysis below — this is what a teammate reads first, and it should stand on its own without anyone having to open the code. At least 500 words. Cover: what you found, the actual numbers, anything that makes you trust or distrust the result, and whether this looks worth extending to more companies.*

---

*(summary goes here)*

## Open question

Before anyone invests in automating a data-freshness signal off SEC filings, we want to know whether that's even a reliable thing to measure — fiscal quarters don't line up with calendar quarters for every company, and "the most recent filing" can quietly be months old. There's a start on checking this for one company below; it needs finishing, plus the write-up above.

## Setup

No API key needed — SEC's EDGAR data is fully public. The only requirement is a descriptive `User-Agent` header identifying you; SEC blocks requests that don't have one. (Try commenting out the header below and re-running the next cell — you'll get a `403 Forbidden`.)

In [ ]:
import requests
import pandas as pd
from datetime import date

# SEC asks that this identify you (name + email) — not strictly enforced, but be a good citizen.
HEADERS = {"User-Agent": "Your Name your.email@example.com"}

# A few real companies with a plain Jan-Dec fiscal year, confirmed to work cleanly with this
# notebook end to end. Pick one, or see the note at the bottom about using any other ticker.
COMPANIES = {
    "XOM": "0000034088",   # Exxon Mobil
    "IBM": "0000051143",   # IBM
    "MCD": "0000063908",   # McDonald's
    "CAT": "0000018230",   # Caterpillar
    "PFE": "0000078003",   # Pfizer
}

TICKER = "XOM"  # <- change this to whichever company you want to investigate
CIK = COMPANIES[TICKER]

## Step 1 — Pull the filing history

One request to EDGAR's submissions API returns this company's entire recent filing history as parallel arrays. Turn it into a table and you already have real data.

In [ ]:
resp = requests.get(f"https://data.sec.gov/submissions/CIK{CIK}.json", headers=HEADERS)
resp.raise_for_status()
recent = resp.json()["filings"]["recent"]

filings = pd.DataFrame(recent)[["form", "filingDate", "reportDate", "accessionNumber"]]
filings["filingDate"] = pd.to_datetime(filings["filingDate"])
filings["reportDate"] = pd.to_datetime(filings["reportDate"])

# Keep just the periodic reports - 10-Ks (annual) and 10-Qs (quarterly)
filings = filings[filings["form"].isin(["10-K", "10-Q"])].sort_values("reportDate").reset_index(drop=True)
filings.tail(10)

## Step 2 — Tag each filing with its fiscal quarter

A filing's `reportDate` is a calendar date (e.g. `2025-06-30`); we need it mapped to a fiscal label (`Q1`-`Q4`/`FY`). The companies above use a plain Jan-Dec fiscal year, so this is calendar-quarter math — not true for every company (Apple, Costco, and Nike all use something else), but out of scope here.

`fiscal_period()` below needs to return `"FY"` for a 10-K, or `"Q1"`/`"Q2"`/`"Q3"`/`"Q4"` for a 10-Q, based on `report_date`'s month.

In [ ]:
def fiscal_period(form: str, report_date: pd.Timestamp) -> str:
    """Return 'FY' for a 10-K, or 'Q1'..'Q4' for a 10-Q, based on report_date's month.
    Assumes a plain January-December fiscal year.
    """
    # TODO: implement this
    ...


filings["fiscal_period"] = filings.apply(lambda r: fiscal_period(r["form"], r["reportDate"]), axis=1)
filings["fiscal_year"] = filings["reportDate"].dt.year
filings["period_label"] = filings["fiscal_year"].astype(str) + "-" + filings["fiscal_period"]
filings[["form", "reportDate", "fiscal_period", "period_label"]].tail(10)

## Step 3 — Find what's missing and how stale the data is

Given the last 8 expected periods (2 fiscal years × Q1/Q2/Q3/FY), which does this company actually have a filing for, and how many days has it been since the most recent filing? Two things are still missing below: the list of uncovered periods, and the age of the most recent filing.

In [ ]:
current_year = date.today().year
expected_periods = [
    f"{y}-{p}"
    for y in [current_year - 2, current_year - 1]
    for p in ["Q1", "Q2", "Q3", "FY"]
]
filed_periods = filings["period_label"].tolist()

# TODO: entries in expected_periods that aren't in filed_periods
missing_periods = ...

# TODO: days between today and the most recent value in filings["filingDate"]
days_since_last_filing = ...

is_stale = days_since_last_filing > 100  # a simple, illustrative threshold - not a real cadence model

print(f"Missing periods: {missing_periods}")
print(f"Days since last filing: {days_since_last_filing} (stale: {is_stale})")

## Step 4 — Pull one real figure

Pull one headline number (revenue) via EDGAR's XBRL API and line it up against the filing table you already built.

In [ ]:
concept = requests.get(
    f"https://data.sec.gov/api/xbrl/companyconcept/CIK{CIK}/us-gaap/Revenues.json",
    headers=HEADERS,
).json()

revenue = pd.DataFrame(concept["units"]["USD"])
revenue["start"] = pd.to_datetime(revenue["start"])
revenue["end"] = pd.to_datetime(revenue["end"])
revenue["duration_days"] = (revenue["end"] - revenue["start"]).dt.days

# A single filing often reports the same tag for more than one period: prior-year
# comparatives, and (for 10-Qs) both "this quarter" and "year-to-date" figures. Keep only
# the row whose duration matches a single quarter (10-Q) or a full year (10-K) - that's
# the one that actually corresponds to this filing's own reportDate.
is_quarter = revenue["duration_days"].between(80, 100)
is_year = revenue["duration_days"].between(350, 380)
revenue = revenue[revenue["form"].isin(["10-K", "10-Q"]) & (is_quarter | is_year)]
revenue = revenue[["accn", "end", "val"]]
revenue.tail(10)

What's left: merge `revenue`'s `val` column onto `filings`, matching **both** `accessionNumber == accn` **and** `reportDate == end` (matching only on accession number isn't enough — see the comment above about why one filing can contain more than one value for the same tag). Rename `val` to `revenue`, and call the result `report`.

In [ ]:
# TODO: merge revenue onto filings (match accessionNumber==accn AND reportDate==end),
# rename val -> revenue, call the result `report`
report = ...

report[["period_label", "reportDate", "revenue"]].tail(10)

## Coverage table

One glance at which periods are actually covered, using `expected_periods`/`filed_periods` from Step 3 — given, should just run.

In [ ]:
coverage = pd.DataFrame({"period": expected_periods})
coverage["status"] = coverage["period"].apply(lambda p: "filed" if p in filed_periods else "missing")


def highlight_missing(row):
    color = "background-color: #f8d7da" if row["status"] == "missing" else "background-color: #d4edda"
    return [color, color]


coverage.style.apply(highlight_missing, axis=1)

## Make one chart

Show revenue by period. A simple bar chart is enough — this doesn't need to be polished.

In [ ]:
# TODO: make one chart. A starting point:
# report.set_index("period_label")["revenue"].plot(kind="bar", title=f"{TICKER} revenue by period")
...

**Exploring a different company?** Steps 1-3 work for any ticker with a plain calendar fiscal year. Step 4's `Revenues` tag isn't used consistently by every company — some (like Amazon) tag revenue differently, and banks especially often don't tag it quarterly at all. If your pick comes back with missing revenue values in Step 4, that's a legitimate real-data finding worth noting, not a bug to hide.

## Wrap up

Save your table, make sure every cell above has been run (so the tables and chart are saved in the notebook itself), then go back and write the Executive Summary at the top — that's what the team will actually read.

In [ ]:
report.to_csv(f"{TICKER}_filing_freshness.csv", index=False)
print("Saved.")